In [1]:
import time

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score
 )
from sklearn.preprocessing import LabelEncoder, StandardScaler

from tensorflow.keras import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Input

test_path = r"C:\AMRITA\Final_Year_Project_zero_day_27\model_selection_zeroday_fyp_27\UNSW_NB15_testing-set.csv"
train_path = r"C:\AMRITA\Final_Year_Project_zero_day_27\model_selection_zeroday_fyp_27\UNSW_NB15_training-set.csv"

test_df = pd.read_csv(test_path)
train_df = pd.read_csv(train_path)

print("Training dataset shape:", train_df.shape)
print("Test dataset shape:", test_df.shape)

Training dataset shape: (82332, 45)
Test dataset shape: (175341, 45)


In [2]:
selected_features = [

    "dur",
    "proto",
    "service",
    "spkts",
    "dpkts",
    "sbytes",
    "dbytes",
    "rate",
    "sttl",
    "dttl",
    "sload",
    "dload",
    "sinpkt",
    "dinpkt",
    "sjit",
    "djit",
    "tcprtt",
    "synack",
    "ackdat",
    "ct_srv_src",
    "ct_state_ttl",
    "ct_dst_ltm",
    "ct_src_ltm",
    "ct_srv_dst"

]

X_test_df = test_df[selected_features].copy()

print(X_test_df.shape)

(175341, 24)


In [4]:
from sklearn.preprocessing import LabelEncoder

proto_encoder = LabelEncoder()
service_encoder = LabelEncoder()

proto_encoder.fit(
    train_df["proto"].astype(str)
 )

service_encoder.fit(
    train_df["service"].astype(str)
 )

proto_mapping = {
    value: index
    for index, value in enumerate(proto_encoder.classes_)
}
service_mapping = {
    value: index
    for index, value in enumerate(service_encoder.classes_)
}

X_test_df["proto"] = (
    X_test_df["proto"].astype(str).map(proto_mapping).fillna(-1)
 )
X_test_df["service"] = (
    X_test_df["service"].astype(str).map(service_mapping).fillna(-1)
 )

print(
    "Unknown test proto values:",
    (X_test_df["proto"] == -1).sum()
 )
print(
    "Unknown test service values:",
    (X_test_df["service"] == -1).sum()
 )

Unknown test proto values: 16
Unknown test service values: 0


In [5]:
X_test_df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

X_train_df = train_df[selected_features].copy()

X_train_df["proto"] = proto_encoder.transform(
    X_train_df["proto"].astype(str)
)

X_train_df["service"] = service_encoder.transform(
    X_train_df["service"].astype(str)
)

X_train_df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

train_medians = X_train_df.median()

X_test_df.fillna(
    train_medians,
    inplace=True
)

,dur,proto,service,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,...,sjit,djit,tcprtt,synack,ackdat,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_ltm,ct_srv_dst
0,0.121478,111.0,0,6,4,258,172,74.087490,252,254,...,30.177547,11.830604,0.000000,0.000000,0.000000,1,0,1,1,1
1,0.649902,111.0,0,14,38,734,42014,78.473372,62,252,...,61.426934,1387.778330,0.000000,0.000000,0.000000,43,1,1,1,6
2,1.623129,111.0,0,8,16,364,13186,14.170161,62,252,...,17179.586860,11420.926230,0.111897,0.061458,0.050439,7,1,2,2,6
3,1.681642,111.0,3,12,12,628,770,13.677108,62,252,...,259.080172,4991.784669,0.000000,0.000000,0.000000,1,1,2,2,1
4,0.449454,111.0,0,10,6,534,268,33.373826,254,252,...,2415.837634,115.807000,0.128381,0.071147,0.057234,43,1,2,2,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175336,0.000009,117.0,2,2,0,114,0,111111.107200,254,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,24,2,24,24,24
175337,0.505762,111.0,0,10,8,620,354,33.612649,254,252,...,3721.068786,120.177727,0.099440,0.036895,0.062545,1,1,1,1,1
175338,0.000009,117.0,2,2,0,114,0,111111.107200,254,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,12,2,3,3,12
175339,0.000009,117.0,2,2,0,114,0,111111.107200,254,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,30,2,30,30,30


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train_df
)

X_test = scaler.transform(
    X_test_df
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (82332, 24)
Testing : (175341, 24)


In [7]:
def evaluate_clustering(
    X_data,
    labels,
    metric_sample_size=10000
):
    
    labels = np.asarray(labels)
    
    # Remove noise labels if any
    valid = labels != -1
    
    X_eval = X_data[valid]
    y_eval = labels[valid]
    
    n_clusters = len(
        np.unique(y_eval)
    )
    
    if n_clusters < 2:
        return {
            "Clusters": n_clusters,
            "Silhouette": np.nan,
            "Davies-Bouldin": np.nan,
            "Calinski-Harabasz": np.nan
        }
    
    # Fixed sampling for faster metric calculation
    if len(X_eval) > metric_sample_size:
        
        rng = np.random.RandomState(42)
        
        idx = rng.choice(
            len(X_eval),
            size=metric_sample_size,
            replace=False
        )
        
        X_eval = X_eval[idx]
        y_eval = y_eval[idx]
    
    return {
        "Clusters": n_clusters,
        
        "Silhouette": silhouette_score(
            X_eval,
            y_eval
        ),
        
        "Davies-Bouldin": davies_bouldin_score(
            X_eval,
            y_eval
        ),
        
        "Calinski-Harabasz": calinski_harabasz_score(
            X_eval,
            y_eval
        )
    }

In [8]:
def build_dec_autoencoder(
    input_dim,
    latent_dim
):
    
    inputs = Input(
        shape=(input_dim,)
    )
    
    # Encoder
    x = Dense(
        32,
        activation="relu"
    )(inputs)
    
    x = Dense(
        16,
        activation="relu"
    )(x)
    
    latent = Dense(
        latent_dim,
        activation="linear",
        name="latent"
    )(x)
    
    # Decoder
    x = Dense(
        16,
        activation="relu"
    )(latent)
    
    x = Dense(
        32,
        activation="relu"
    )(x)
    
    outputs = Dense(
        input_dim,
        activation="linear"
    )(x)
    
    autoencoder = Model(
        inputs,
        outputs,
        name="DEC_Autoencoder"
    )
    
    encoder = Model(
        inputs,
        latent,
        name="DEC_Encoder"
    )
    
    return autoencoder, encoder

In [9]:
def pretrain_dec_autoencoder(
    X_data,
    latent_dim,
    epochs=20,
    batch_size=256
):
    
    input_dim = X_data.shape[1]
    
    autoencoder, encoder = build_dec_autoencoder(
        input_dim=input_dim,
        latent_dim=latent_dim
    )
    
    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="mse"
    )
    
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )
    
    history = autoencoder.fit(
        X_data,
        X_data,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.1,
        callbacks=[early_stopping],
        verbose=0
    )
    
    return (
        autoencoder,
        encoder,
        history
    )

In [10]:
class DECLayer(tf.keras.layers.Layer):
    
    def __init__(
        self,
        n_clusters,
        alpha=1.0,
        **kwargs
    ):
        
        super().__init__(**kwargs)
        
        self.n_clusters = n_clusters
        self.alpha = alpha
    
    def build(self, input_shape):
        
        input_dim = int(
            input_shape[-1]
        )
        
        self.clusters = self.add_weight(
            shape=(
                self.n_clusters,
                input_dim
            ),
            initializer="glorot_uniform",
            trainable=True,
            name="cluster_centers"
        )
        
        super().build(input_shape)
    
    def call(self, inputs):
        
        # Squared distance from every
        # sample to every cluster center
        
        distances = tf.reduce_sum(
            tf.square(
                tf.expand_dims(
                    inputs,
                    axis=1
                )
                - self.clusters
            ),
            axis=2
        )
        
        # Student's t-distribution
        
        q = 1.0 / (
            1.0 + distances / self.alpha
        )
        
        q = tf.pow(
            q,
            (self.alpha + 1.0) / 2.0
        )
        
        q = q / tf.reduce_sum(
            q,
            axis=1,
            keepdims=True
        )
        
        return q

In [11]:
def dec_target_distribution(q):
    
    weight = (
        q ** 2
    ) / (
        np.sum(q, axis=0) + 1e-10
    )
    
    target = (
        weight.T
        /
        (
            np.sum(
                weight,
                axis=1
            ) + 1e-10
        )
    ).T
    
    return target

In [12]:
def train_dec(
    X_data,
    latent_dim=4,
    n_clusters=10,
    pretrain_epochs=20,
    dec_iterations=500,
    update_interval=50,
    batch_size=256,
    learning_rate=0.001,
    tol=0.001,
    random_state=42
):
    
    start_time = time.time()
    
    # ==================================================
    # STEP 1 — Pretrain Autoencoder
    # ==================================================
    
    (
        autoencoder,
        encoder,
        history
    ) = pretrain_dec_autoencoder(
        X_data,
        latent_dim=latent_dim,
        epochs=pretrain_epochs,
        batch_size=batch_size
    )
    
    # ==================================================
    # STEP 2 — Obtain latent representation
    # ==================================================
    
    latent_data = encoder.predict(
        X_data,
        batch_size=batch_size,
        verbose=0
    )
    
    # ==================================================
    # STEP 3 — Initialize cluster centers using K-Means
    # ==================================================
    
    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=10
    )
    
    initial_labels = kmeans.fit_predict(
        latent_data
    )
    
    initial_centers = (
        kmeans.cluster_centers_
    )
    
    # ==================================================
    # STEP 4 — Create DEC clustering layer
    # ==================================================
    
    clustering_layer = DECLayer(
        n_clusters=n_clusters,
        alpha=1.0,
        name="clustering"
    )
    
    # Build the layer
    _ = clustering_layer(
        encoder.output
    )
    
    # Initialize cluster centers
    clustering_layer.set_weights(
        [initial_centers]
    )
    
    # ==================================================
    # STEP 5 — Build DEC model
    # ==================================================
    
    dec_model = Model(
        inputs=encoder.input,
        outputs=clustering_layer(
            encoder.output
        )
    )
    
    dec_model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="kld"
    )
    
    # ==================================================
    # STEP 6 — Initial probability distribution
    # ==================================================
    
    q = dec_model.predict(
        X_data,
        batch_size=batch_size,
        verbose=0
    )
    
    p = dec_target_distribution(q)
    
    previous_labels = q.argmax(
        axis=1
    )
    
    # ==================================================
    # STEP 7 — DEC optimization
    # ==================================================
    
    final_iteration = 0
    
    for iteration in range(
        dec_iterations
    ):
        
        final_iteration = iteration
        
        # ----------------------------------------------
        # Update target distribution
        # ----------------------------------------------
        
        if iteration % update_interval == 0:
            
            q = dec_model.predict(
                X_data,
                batch_size=batch_size,
                verbose=0
            )
            
            p = dec_target_distribution(q)
            
            current_labels = q.argmax(
                axis=1
            )
            
            # Check convergence
            if iteration > 0:
                
                delta_label = np.mean(
                    current_labels
                    != previous_labels
                )
                
                if delta_label < tol:
                    
                    print(
                        f"Converged at "
                        f"iteration {iteration}"
                    )
                    
                    break
            
            previous_labels = current_labels
        
        # ----------------------------------------------
        # Mini-batch training
        # ----------------------------------------------
        
        batch_start = (
            iteration * batch_size
        ) % len(X_data)
        
        batch_end = min(
            batch_start + batch_size,
            len(X_data)
        )
        
        batch_indices = np.arange(
            batch_start,
            batch_end
        )
        
        dec_model.train_on_batch(
            X_data[batch_indices],
            p[batch_indices]
        )
    
    # ==================================================
    # STEP 8 — Final prediction
    # ==================================================
    
    q_final = dec_model.predict(
        X_data,
        batch_size=batch_size,
        verbose=0
    )
    
    final_labels = q_final.argmax(
        axis=1
    )
    
    final_latent = encoder.predict(
        X_data,
        batch_size=batch_size,
        verbose=0
    )
    
    training_time = (
        time.time()
        - start_time
    )
    
    return {
        "model": dec_model,
        "encoder": encoder,
        "latent": final_latent,
        "labels": final_labels,
        "training_time": training_time,
        "iterations": final_iteration + 1
    }

In [13]:
# Train DEC on the complete training set and evaluate unseen test samples.
DEC_LATENT_DIM = 2
DEC_CLUSTERS = 5

dec_start = time.time()
dec_result = train_dec(
    X_train,
    latent_dim=DEC_LATENT_DIM,
    n_clusters=DEC_CLUSTERS,
    pretrain_epochs=20,
    dec_iterations=500,
    update_interval=50,
    batch_size=256,
    learning_rate=0.001,
    tol=0.001,
    random_state=42
)

test_probabilities = dec_result["model"].predict(
    X_test,
    batch_size=256,
    verbose=0
)
test_labels = test_probabilities.argmax(axis=1)
test_latent = dec_result["encoder"].predict(
    X_test,
    batch_size=256,
    verbose=0
)

test_metrics = evaluate_clustering(
    test_latent,
    test_labels,
    metric_sample_size=10000
)

print("=" * 60)
print("DEC HELD-OUT TEST RESULT")
print("=" * 60)
print("Latent dimension:", DEC_LATENT_DIM)
print("Clusters:", DEC_CLUSTERS)
print("Training samples:", len(X_train))
print("Test samples:", len(X_test))
print("Training time:", round(dec_result["training_time"], 2), "seconds")
print("Total evaluation time:", round(time.time() - dec_start, 2), "seconds")
print("Predicted cluster counts:")
print(pd.Series(test_labels).value_counts().sort_index())
print("Silhouette:", round(test_metrics["Silhouette"], 4))
print("Davies-Bouldin:", round(test_metrics["Davies-Bouldin"], 4))
print("Calinski-Harabasz:", round(test_metrics["Calinski-Harabasz"], 2))

# Cluster IDs are arbitrary, so ARI/NMI are used instead of accuracy.
if "attack_cat" in test_df.columns:
    true_labels = test_df["attack_cat"].astype(str)
elif "label" in test_df.columns:
    true_labels = test_df["label"].astype(str)
else:
    true_labels = None

if true_labels is not None:
    true_encoder = LabelEncoder()
    true_labels_encoded = true_encoder.fit_transform(true_labels)
    print("ARI:", round(
        adjusted_rand_score(true_labels_encoded, test_labels),
        4
    ))
    print("NMI:", round(
        normalized_mutual_info_score(true_labels_encoded, test_labels),
        4
    ))
else:
    print("No label column found; ARI/NMI were not calculated.")


DEC HELD-OUT TEST RESULT
Latent dimension: 2
Clusters: 5
Training samples: 82332
Test samples: 175341
Training time: 80.91 seconds
Total evaluation time: 91.99 seconds
Predicted cluster counts:
0    40527
1    89612
2    29443
3    15631
4      128
Name: count, dtype: int64
Silhouette: 0.7402
Davies-Bouldin: 0.502
Calinski-Harabasz: 19401.49
ARI: 0.2664
NMI: 0.2461
